# Supervisor-Worker Pattern
## One Director, Many Workers — Hierarchical Agent Control
⏱ ~12 min

The **Supervisor-Worker pattern** is one of the most important architectures in production multi-agent systems. A central supervisor LLM classifies each incoming task and routes it to the specialist subgraph best equipped to handle it. Workers never communicate with each other — they receive a task from the supervisor and return a result. The supervisor then optionally aggregates, grades, or re-routes.

By the end of this session you will understand *why* this architecture exists, *how* structured output makes routing safe and auditable, and *how* to build, extend, and debug a supervisor-worker graph in LangGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Multi-agent taxonomy and when to use a supervisor |
| 2 | **Structured Routing** — Why `with_structured_output` beats string parsing |
| 3 | **State & Schema** — `TypedDict` state, `Pydantic` routing schema |
| 4 | **Worker Factory** — Building specialized workers from a single function |
| 5 | **Graph Assembly** — Wiring supervisor + workers in LangGraph |
| 6 | **Introspection & Debugging** — Inspecting routing decisions and state |
| 7 | **Extensions** — 4th worker, feedback loops, confidence scoring |
| ★ | **Advanced** — Parallel workers via `Send`, result aggregation |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing requirements from `requirements.txt`
- `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Wu, Q., Bansal, G., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442  
> LangGraph multi-agent docs: https://langchain-ai.github.io/langgraph/concepts/multi_agent/  
> Pydantic structured output: https://python.langchain.com/docs/concepts/structured_outputs/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "pydantic",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
from collections import Counter
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: echo 'OPENAI_API_KEY=sk-...' >> .env | "
    "Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True ({key[:8]}...)")

---

## Part 1 — Concepts: Multi-Agent Taxonomy

---

### Why Multi-Agent at All?

A single LLM handles most tasks well — but breaks down when:

1. **Task diversity is high** — one prompt cannot be optimized for research, summarization, AND analysis simultaneously.
2. **Quality demands role specialization** — a researcher and an analyst have different system prompts, temperatures, and tools.
3. **Scale requires parallelism** — multiple workers can run concurrently on independent sub-tasks.
4. **Auditability matters** — you need to know *why* the system chose a particular path.

---

### Multi-Agent Topology Comparison

| Topology | Routing | Communication | Best for |
|----------|---------|---------------|----------|
| **Supervisor-Worker** (this example) | Central supervisor classifies & routes | One-way: supervisor → worker → result | Task classification, specialist dispatch |
| **Peer-to-peer** (e.g., AutoGen) | Any agent can message any other | Bidirectional, conversational | Collaborative reasoning, debate |
| **Pipeline** | Fixed sequential order | Output of step N → input of step N+1 | ETL, multi-stage processing |
| **Hierarchical** | Supervisors can have sub-supervisors | Tree-structured delegation | Large teams, complex decomposition |
| **Swarm** | Agents self-select tasks | Shared queue or state | Massive parallelism, homogeneous tasks |

---

### When to Choose Supervisor-Worker

Choose this pattern when:
- You have **3–10 distinct task types** that map to specialist workers
- The routing decision can be made **from the task description alone** (no inter-agent communication needed first)
- You want **explainable routing** — a `rationale` field you can log, audit, or surface in a UI
- Workers are **independent** — they do not need to talk to each other

---

### Architecture Diagram

```
                         ┌─────────────────────────────────────────┐
                         │           SUPERVISOR NODE               │
                         │                                         │
   User Task             │  router_llm.with_structured_output()    │
  ──────────►            │  → WorkerRoute(worker='researcher',     │
                         │               rationale='needs facts')  │
                         └──────────────┬──────────────────────────┘
                                        │
                         conditional_edges(route fn)
                                        │
               ┌────────────────────────┼────────────────────────┐
               │                        │                        │
               ▼                        ▼                        ▼
   ┌─────────────────────┐  ┌─────────────────────┐  ┌─────────────────────┐
   │   RESEARCHER NODE   │  │  SUMMARIZER NODE    │  │   ANALYST NODE      │
   │                     │  │                     │  │                     │
   │ system: "provide    │  │ system: "give a     │  │ system: "provide    │
   │  3-5 factual        │  │  3-bullet summary" │  │  pros/cons/         │
   │  points"            │  │                     │  │  recommendation"   │
   └──────────┬──────────┘  └──────────┬──────────┘  └──────────┬──────────┘
              │                        │                         │
              └────────────────────────┼─────────────────────────┘
                                       │
                                       ▼
                                      END
                               worker_result in state
```

> **Key insight:** The supervisor only runs once per task. It reads the task, emits a structured `WorkerRoute`, and the graph router sends control to exactly one worker subgraph. The worker produces the final `worker_result` and the graph terminates.

---

## Part 2 — Structured Routing: Why `with_structured_output` Wins

---

### The Routing Problem

Multi-agent routing requires the supervisor LLM to name a valid worker. The naive approach — parsing free text — is fragile:

```python
# FRAGILE: string parsing
response = llm.invoke("Which worker should handle this?")
text = response.content.lower()
if 'researcher' in text:
    route = 'researcher'
elif 'summarizer' in text:
    route = 'summarizer'
else:
    route = 'researcher'  # silent default hides routing failures
```

Problems:
- LLM may say "I would use the **research** worker" — `researcher` not in response, falls back silently
- No explanation for *why* the route was chosen
- Adding a new worker means updating string-matching in multiple places

---

### The Structured Output Solution

```python
# ROBUST: structured output with Pydantic
class WorkerRoute(BaseModel):
    worker: Literal['researcher', 'summarizer', 'analyst']  # enforced
    rationale: str                                           # auditable

router_llm = llm.with_structured_output(WorkerRoute)
route = router_llm.invoke(prompt)   # raises ValidationError if invalid
route.worker    # guaranteed to be one of the three values
route.rationale # explains the decision
```

---

### Comparison Table

| Property | String parsing | `with_structured_output` |
|----------|---------------|--------------------------|
| Valid worker guaranteed | No — silent fallback | Yes — `Literal` enforced by Pydantic |
| Explainability | None | `rationale` field |
| Error visibility | Silent wrong route | `ValidationError` raised immediately |
| Add a new worker | Update string matching | Add to `Literal`, add node |
| Schema is self-documenting | No | Yes — type hint IS the contract |

In [ ]:
# ─── 2-A: Define WorkerRoute and observe structured output ────────────────────
# The Literal type restricts the LLM to exactly these three worker names.
# Pydantic raises ValidationError if the model returns anything else.


class WorkerRoute(BaseModel):
    """Routing decision emitted by the supervisor LLM."""

    worker: Literal["researcher", "summarizer", "analyst"]
    rationale: str  # explains WHY this worker was chosen — log it!


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
router_llm = llm.with_structured_output(WorkerRoute)

SUPERVISOR_PROMPT = (
    "You are a supervisor. Classify this task to the best worker: "
    "researcher (needs facts/research), summarizer (needs condensed info), "
    "analyst (needs structured analysis). Task: {task}"
)

# Test: each task should route to a different worker
test_tasks = [
    "What are the key facts about quantum entanglement?",
    "Give me a short overview of the Python docs on decorators.",
    "Should I use PostgreSQL or MongoDB for my new SaaS?",
]

print("Routing decisions:\n")
for task in test_tasks:
    result = router_llm.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=task))])
    print(f"  Task:      {task[:60]}")
    print(f"  Worker:    {result.worker}")
    print(f"  Rationale: {result.rationale[:100]}")
    print()

In [ ]:
# ─── 2-B: Observe routing on ambiguous tasks ──────────────────────────────────
# The model still commits to one valid worker — it cannot return an invalid name.

ambiguous_tasks = [
    "Tell me about machine learning.",  # could be researcher OR summarizer
    "Is Python a good language?",       # research + analysis blend
]

print("Ambiguous task routing:\n")
for task in ambiguous_tasks:
    result = router_llm.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=task))])
    print(f"  Task:      {task}")
    print(f"  Worker:    {result.worker}")
    print(f"  Rationale: {result.rationale}")
    print()

print("Observation: even ambiguous tasks receive a valid, justified routing decision.")

---

## Part 3 — State and Schema

---

### LangGraph State

Every LangGraph graph has a **state** — a `TypedDict` that flows through all nodes. Each node receives the full current state and returns a dict with only the fields it wants to update. Fields not returned are left unchanged.

```
Initial state:
  { task: "...", worker_type: "", rationale: "", worker_result: "" }
                    │
              supervisor node
                    │
  Updates: { worker_type: "researcher", rationale: "needs facts" }
                    │
              researcher node
                    │
  Updates: { worker_result: "1. Quantum computing uses..." }
                    │
                   END
  Final state:
  { task: "...", worker_type: "researcher", rationale: "...", worker_result: "..." }
```

### State Field Reference

| Field | Type | Set by | Purpose |
|-------|------|--------|--------|
| `task` | `str` | caller (`app.invoke`) | The original user task — read-only after entry |
| `worker_type` | `str` | `supervisor` node | Which worker to route to |
| `rationale` | `str` | `supervisor` node | Explanation for the routing decision |
| `worker_result` | `str` | worker node | Final output from the specialist worker |

In [ ]:
# ─── 3-A: Define the state schema ─────────────────────────────────────────────
# TypedDict gives LangGraph the field names and types.
# All fields must be present in the initial state dict passed to app.invoke().


class SupervisorState(TypedDict):
    task: str           # the user's request — set once at entry
    worker_type: str    # which specialist was selected by the supervisor
    rationale: str      # why the supervisor chose this worker
    worker_result: str  # the specialist's output


# The empty initial state template — used for every app.invoke() call
EMPTY_STATE: SupervisorState = {
    "task": "",
    "worker_type": "",
    "rationale": "",
    "worker_result": "",
}

print("State schema: SupervisorState")
for field, annotation in SupervisorState.__annotations__.items():
    print(f"  {field:16s}: {annotation.__name__}")

In [ ]:
# ─── 3-B: Simulate partial state updates from each node ───────────────────────
# Nodes return PARTIAL dicts — only the fields they touch.
# LangGraph merges these updates into the running state dict.

print("Node return value examples (partial state updates):\n")

# Supervisor node returns routing decision
supervisor_update = {
    "worker_type": "researcher",
    "rationale": "Task asks for latest developments — factual research is required.",
}
print("supervisor node returns:")
for k, v in supervisor_update.items():
    print(f"  {k}: {repr(v[:60])}")

print()

# Worker node returns result
worker_update = {
    "worker_result": "1. IBM announced a 1000-qubit processor in 2023...\n2. ...",
}
print("researcher node returns:")
for k, v in worker_update.items():
    print(f"  {k}: {repr(v[:80])}")

print()
print("Fields NOT in the return dict (task, worker_type after worker runs) are unchanged.")

---

## Part 4 — The Worker Factory

---

### Pattern: Factory Function

All three workers share identical structure — they receive the task from state, call the LLM with a specialized system prompt, and write the result to state. The only difference is the system prompt.

A **factory function** `make_worker(worker_type)` generates each worker as a closure, capturing the correct `worker_type` string. This avoids copy-pasting three nearly-identical functions:

```python
# WITHOUT factory — repetitive and error-prone:
def researcher_worker(state): ...
def summarizer_worker(state): ...
def analyst_worker(state): ...

# WITH factory — DRY and extensible:
for wt in WORKER_PROMPTS:
    graph.add_node(wt, make_worker(wt))   # one line per worker
```

---

### Worker Prompt Design

| Worker | System prompt strategy | Output style |
|--------|----------------------|-------------|
| `researcher` | "Provide 3-5 factual, well-sourced points" | Numbered facts |
| `summarizer` | "Give a concise 3-bullet summary of key points" | Bullet list |
| `analyst` | "Provide a structured analysis with pros/cons/recommendation" | Structured sections |

In [ ]:
# ─── 4-A: Define worker prompts and the factory function ──────────────────────

WORKER_PROMPTS = {
    "researcher": (
        "You are a researcher. Provide 3-5 factual, well-sourced points about: {task}"
    ),
    "summarizer": (
        "You are a summarizer. Give a concise 3-bullet summary of key points about: {task}"
    ),
    "analyst": (
        "You are an analyst. Provide a structured analysis with pros/cons/recommendation "
        "for: {task}"
    ),
}


def make_worker(worker_type: str):
    """Factory: returns a LangGraph node function for the given worker type."""

    def worker(state: SupervisorState) -> dict:
        prompt = WORKER_PROMPTS[worker_type].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"worker_result": result.content}

    worker.__name__ = f"{worker_type}_worker"  # meaningful name for debugging
    return worker


# Inspect the generated functions
for wt in WORKER_PROMPTS:
    fn = make_worker(wt)
    print(f"  make_worker('{wt}') → function '{fn.__name__}'")

print()
print("Each closure captures its worker_type — same code structure, different prompt.")

In [ ]:
# ─── 4-B: Test each worker independently (before wiring the graph) ────────────
# Always test nodes in isolation before connecting them.

test_state: SupervisorState = {
    "task": "the history of the internet",
    "worker_type": "",
    "rationale": "",
    "worker_result": "",
}

print("Testing each worker node in isolation:\n")
for wt in WORKER_PROMPTS:
    fn = make_worker(wt)
    update = fn(test_state)
    preview = update["worker_result"][:200].replace("\n", " ")
    print(f"[{wt}]")
    print(f"  {preview}...")
    print()

---

## Part 5 — Graph Assembly

---

### LangGraph Building Blocks

| Concept | API | Role |
|---------|-----|------|
| **Node** | `graph.add_node(name, fn)` | A Python function that reads state and returns updates |
| **Entry point** | `graph.set_entry_point(name)` | The first node to run when `app.invoke()` is called |
| **Direct edge** | `graph.add_edge(a, b)` | Always go from node a to node b |
| **Conditional edge** | `graph.add_conditional_edges(a, fn, map)` | Run `fn(state)` → use the return value to look up the next node in `map` |
| **Compile** | `graph.compile()` | Validates the graph and returns a runnable |

---

### The Routing Function

```python
def route(state: SupervisorState) -> str:
    return state["worker_type"]   # returns 'researcher', 'summarizer', or 'analyst'
```

This function bridges `with_structured_output` (which writes `worker_type` to state) and `add_conditional_edges` (which reads from state to pick the next node). The mapping dict `{wt: wt for wt in WORKER_PROMPTS}` maps each worker name string to the node of the same name.

In [ ]:
# ─── 5-A: Define the supervisor node and routing function ─────────────────────


def supervisor(state: SupervisorState) -> dict:
    """Read the task, invoke the router LLM, write routing decision to state."""
    prompt = SUPERVISOR_PROMPT.format(task=state["task"])
    result = router_llm.invoke([HumanMessage(content=prompt)])
    print(f"  [supervisor] → {result.worker}")
    print(f"  [rationale]  → {result.rationale[:80]}")
    return {"worker_type": result.worker, "rationale": result.rationale}


def route(state: SupervisorState) -> str:
    """Routing function: reads worker_type from state, returns node name to visit."""
    return state["worker_type"]


print("Supervisor node and routing function defined.")

In [ ]:
# ─── 5-B: Assemble and compile the graph ─────────────────────────────────────


def create_workflow():
    graph = StateGraph(SupervisorState)

    # Add supervisor node
    graph.add_node("supervisor", supervisor)

    # Add one worker node per worker type — factory generates the function
    for wt in WORKER_PROMPTS:
        graph.add_node(wt, make_worker(wt))
        graph.add_edge(wt, END)  # all workers terminate at END

    # Entry point and conditional routing
    graph.set_entry_point("supervisor")
    graph.add_conditional_edges(
        "supervisor",
        route,
        {wt: wt for wt in WORKER_PROMPTS},  # identity mapping: name → node
    )

    return graph.compile()


app = create_workflow()
print("Graph compiled successfully.")
print(f"Nodes: {list(app.graph.nodes.keys())}")

In [ ]:
# ─── 5-C: Run the full pipeline on sample tasks ───────────────────────────────

SAMPLE_TASKS = [
    "Research the latest developments in quantum computing.",
    "Summarize the key points of the React documentation.",
    "Analyze the pros and cons of microservices vs monoliths.",
]

results = []
for task in SAMPLE_TASKS:
    print(f"\nTASK: {task}")
    print("-" * 60)
    result = app.invoke({**EMPTY_STATE, "task": task})
    results.append(result)
    print(f"\nWorker: {result['worker_type']}")
    print(f"Result (preview):\n{result['worker_result'][:300]}")
    print()

---

## Part 6 — Introspection and Debugging

---

### What to Check When Routing Goes Wrong

| Symptom | Likely cause | Debug step |
|---------|-------------|------------|
| Wrong worker chosen | Supervisor prompt too vague | Print `rationale` field; rewrite worker descriptions |
| `ValidationError` on routing | LLM output does not match `Literal` | Log the raw LLM output before `with_structured_output` |
| Empty `worker_result` | Worker node errored silently | Test worker node in isolation (Part 4-B pattern) |
| Ambiguous tasks always go to same worker | Supervisor prompt biased | Add negative examples to supervisor prompt |
| State fields missing at END | Node returned wrong keys | Print full final state dict |

In [ ]:
# ─── 6-A: Inspect all fields of the final state ───────────────────────────────
# Always print the full state dict when debugging — partial views hide problems.

debug_task = "Should I learn Rust or Go for systems programming?"
debug_result = app.invoke({**EMPTY_STATE, "task": debug_task})

print("Full final state:\n")
for key, value in debug_result.items():
    display = value if len(str(value)) < 120 else str(value)[:120] + "..."
    print(f"  {key:16s}: {display}")

In [ ]:
# ─── 6-B: Batch routing audit — check distribution across workers ─────────────
# Run a batch of diverse tasks and print the routing distribution.
# If one worker is always chosen, the supervisor prompt needs tuning.

audit_tasks = [
    "What are the key facts about the Renaissance?",
    "Give me a brief overview of Docker.",
    "Pros and cons of remote work for software teams.",
    "Research the history of machine learning.",
    "Summarize the main points of the Agile Manifesto.",
    "Analyze whether TypeScript is worth adopting for a small startup.",
    "What facts do we know about dark matter?",
    "Condense the key ideas from the 12-factor app methodology.",
    "Should we use Redis or Memcached for our caching layer?",
]

routing_log = []
for task in audit_tasks:
    result = router_llm.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=task))])
    routing_log.append((task, result.worker, result.rationale))

# Print routing table
print(f"{'Task':<55} {'Worker':<12} Rationale (first 50 chars)")
print("-" * 120)
for task, worker, rationale in routing_log:
    print(f"  {task[:53]:<55} {worker:<12} {rationale[:50]}")

# Distribution summary
counts = Counter(w for _, w, _ in routing_log)
print(f"\nRouting distribution: {dict(counts)}")
print("Healthy output: all three workers appear at least once.")

---

## Part 7 — Extensions

---

### Exercise 1 — Add a 4th Worker: `coder`

The current graph handles research, summarization, and analysis. Many real tasks require code. Extend the graph with a `coder` worker:

1. Add `'coder'` to `WORKER_PROMPTS` with a prompt like `"Write a short Python script that demonstrates: {task}"`
2. Add `'coder'` to the `Literal` in `WorkerRoute`
3. Re-run `create_workflow()` — the loop handles node creation automatically
4. Test with `"Write a script that reads a CSV file and prints the first 5 rows"`

**Starter below — fill in the TODOs:**

In [ ]:
# ── Exercise 1 Starter — Add a 4th worker ─────────────────────────────────────

# TODO 1: Add 'coder' to the prompts dict
WORKER_PROMPTS_EX1 = {
    "researcher": WORKER_PROMPTS["researcher"],
    "summarizer": WORKER_PROMPTS["summarizer"],
    "analyst": WORKER_PROMPTS["analyst"],
    # "coder": "TODO: write a system prompt for the coder worker",
}

# TODO 2: Update the Literal to include 'coder'
# class WorkerRouteEx1(BaseModel):
#     worker: Literal["researcher", "summarizer", "analyst", "coder"]
#     rationale: str

# TODO 3: Build router_llm_ex1 = llm.with_structured_output(WorkerRouteEx1)

# TODO 4: Rebuild the graph using WORKER_PROMPTS_EX1 and an updated supervisor prompt
# Hint: the loop `for wt in WORKER_PROMPTS_EX1` will pick up the coder automatically

# TODO 5: Test with a code-oriented task:
# result = app_ex1.invoke({**EMPTY_STATE, "task": "Write a script that reads a CSV and prints the first 5 rows."})
# print(result["worker_result"])

print("Exercise 1: uncomment the TODOs above and complete the implementation.")

---

### Exercise 2 — Add a Confidence Score

Add a `confidence: float` field (0.0–1.0) to `WorkerRoute`. If the supervisor's confidence is below `0.6`, log a warning and optionally re-route to a fallback worker.

**What to change:**
1. Add `confidence: float` to the `WorkerRoute` Pydantic model
2. Add `confidence` to `SupervisorState` as a `float` field
3. In the `supervisor` node, write `confidence` to state
4. In the routing function, check `state["confidence"]` — if below threshold, route to `"researcher"` as a safe default

**Starter below:**

In [ ]:
# ── Exercise 2 Starter — Confidence scoring ───────────────────────────────────

# TODO 1: Add confidence field to WorkerRoute
# class WorkerRouteEx2(BaseModel):
#     worker: Literal["researcher", "summarizer", "analyst"]
#     rationale: str
#     confidence: float  # 0.0 = uncertain, 1.0 = certain

# TODO 2: Add confidence to state
# class SupervisorStateEx2(TypedDict):
#     task: str
#     worker_type: str
#     rationale: str
#     confidence: float
#     worker_result: str

# TODO 3: Update supervisor node to write confidence
# def supervisor_ex2(state: SupervisorStateEx2) -> dict:
#     result = router_llm_ex2.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=state["task"]))])
#     if result.confidence < 0.6:
#         print(f"  [WARNING] Low confidence ({result.confidence:.2f}) — check routing")
#     return {"worker_type": result.worker, "rationale": result.rationale, "confidence": result.confidence}

# TODO 4: Update route function to fall back on low confidence
# def route_ex2(state: SupervisorStateEx2) -> str:
#     if state["confidence"] < 0.6:
#         return "researcher"  # safe fallback
#     return state["worker_type"]

print("Exercise 2: uncomment and complete the confidence scoring extension.")

---

### Exercise 3 — Feedback Loop

After a worker produces a result, route back to the supervisor with a grading instruction. If the result scores below a threshold, re-route to a different worker.

**Design sketch:**

```
START → supervisor → worker → grader → END           (if grade >= 7)
                                  │
                                  └──► supervisor → different worker  (if grade < 7)
```

**What to add:**
1. Add `grade: int` and `attempts: int` to `SupervisorState`
2. Add a `grader` node that calls the LLM with `"Grade this output 1-10: {worker_result}"`
3. Add a conditional edge from `grader`: if `grade >= 7` go to `END`; else go back to `supervisor`
4. In `supervisor`, if `attempts > 0`, use the previous `worker_type` as a signal to try a different worker
5. Add a safety limit: if `attempts >= 2`, force `END` to prevent infinite loops

In [ ]:
# ── Exercise 3 Starter — Feedback loop ───────────────────────────────────────

# TODO: Add grade and attempts to state
# class SupervisorStateEx3(TypedDict):
#     task: str
#     worker_type: str
#     rationale: str
#     worker_result: str
#     grade: int       # 1-10 quality score from grader
#     attempts: int    # number of worker invocations so far

# TODO: Grader node using structured output
# class GradeOutput(BaseModel):
#     score: int        # 1-10
#     feedback: str     # what to improve
#
# grader_llm = llm.with_structured_output(GradeOutput)
#
# def grader(state: SupervisorStateEx3) -> dict:
#     result = grader_llm.invoke([
#         HumanMessage(content=(
#             f"Grade this output 1-10 for quality and completeness.\n"
#             f"Task: {state['task']}\nOutput: {state['worker_result']}"
#         ))
#     ])
#     print(f"  [grader] score={result.score}: {result.feedback[:60]}")
#     return {"grade": result.score}

# TODO: Routing logic with feedback loop
# def grade_route(state: SupervisorStateEx3) -> str:
#     if state["grade"] >= 7 or state["attempts"] >= 2:
#         return END
#     return "supervisor"   # retry

print("Exercise 3: uncomment and complete the feedback loop design.")

---

## Part 8 ★ — Advanced: Parallel Workers via `Send`

---

### When to Use Parallel Workers

The base pattern routes to **one** worker. Sometimes you want **multiple** workers to run simultaneously on the same task — for example, getting both a research summary AND an analysis, then letting a final aggregator node synthesize them.

LangGraph's `Send` API enables this. The routing function returns a **list of `Send` objects** instead of a string:

```python
from langgraph.types import Send

def fan_out(state: SupervisorState) -> list[Send]:
    return [
        Send("researcher", state),
        Send("analyst", state),
    ]

graph.add_conditional_edges("supervisor", fan_out)
```

Both worker nodes execute concurrently. An `aggregator` node then receives both results via a list-accumulating state field.

### Parallel vs Single-Worker Trade-offs

| | Single Worker | Parallel Workers |
|-|--------------|------------------|
| Cost | 1 LLM call | N LLM calls |
| Latency | Sequential | Concurrent (wall-clock = slowest worker) |
| Output quality | Specialized | Multi-perspective |
| State complexity | Simple | Requires reducer (list accumulation) |
| Best for | Clear task type | Complex tasks needing multiple views |

In [ ]:
# ─── 8-A: Parallel workers with Send (fan-out pattern) ────────────────────────
# Requires LangGraph >= 0.2 for Send support.
# State uses Annotated[list, operator.add] so each worker APPENDS rather than overwrites.

import operator
from typing import Annotated

try:
    from langgraph.types import Send

    class ParallelState(TypedDict):
        task: str
        # Annotated with operator.add means each worker appends to this list.
        # LangGraph merges parallel writes safely using the reducer.
        worker_results: Annotated[list, operator.add]

    def parallel_researcher(state: ParallelState) -> dict:
        prompt = WORKER_PROMPTS["researcher"].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"worker_results": [{"worker": "researcher", "result": result.content}]}

    def parallel_analyst(state: ParallelState) -> dict:
        prompt = WORKER_PROMPTS["analyst"].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"worker_results": [{"worker": "analyst", "result": result.content}]}

    def fan_out(state: ParallelState) -> list:
        """Dispatch to both researcher and analyst in parallel."""
        return [
            Send("researcher", state),
            Send("analyst", state),
        ]

    def aggregator(state: ParallelState) -> dict:
        """Synthesize results from all workers."""
        print("\nAggregated results:\n")
        for entry in state["worker_results"]:
            print(f"=== {entry['worker'].upper()} ===")
            print(entry["result"][:300])
            print()
        return {}

    parallel_graph = StateGraph(ParallelState)
    parallel_graph.add_node("researcher", parallel_researcher)
    parallel_graph.add_node("analyst", parallel_analyst)
    parallel_graph.add_node("aggregator", aggregator)
    parallel_graph.set_conditional_entry_point(
        fan_out, {"researcher": "researcher", "analyst": "analyst"}
    )
    parallel_graph.add_edge("researcher", "aggregator")
    parallel_graph.add_edge("analyst", "aggregator")
    parallel_graph.add_edge("aggregator", END)
    parallel_app = parallel_graph.compile()

    parallel_result = parallel_app.invoke({
        "task": "Evaluate the trade-offs of using LLMs in production.",
        "worker_results": [],
    })
    print(f"Total worker results collected: {len(parallel_result['worker_results'])}")

except Exception as e:
    print(f"Parallel Send demo requires LangGraph >= 0.2: {e}")

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions, not the only correct approaches.

---

### Exercise 1 Answer — 4th Worker: `coder`

**What changes:** Add `'coder'` to `WORKER_PROMPTS`, update the `Literal` in `WorkerRoute`, rebuild the router LLM, and call `create_workflow()` again. The loop `for wt in WORKER_PROMPTS` picks up the new worker automatically — no other graph wiring changes needed.

**What good output looks like:** A task like "write a Python function that sorts a list of dicts by a key" should route to `coder`, not `researcher`. The `rationale` field should mention "code", "implementation", or "script".

**Common mistake:** Forgetting to update the `Literal` in `WorkerRoute`. If you add the prompt but not the Literal, the router LLM will never emit `'coder'` — Pydantic constrains the output to the declared values.

In [ ]:
# ── Exercise 1 Answer ─────────────────────────────────────────────────────────

WORKER_PROMPTS_V2 = {
    "researcher": WORKER_PROMPTS["researcher"],
    "summarizer": WORKER_PROMPTS["summarizer"],
    "analyst": WORKER_PROMPTS["analyst"],
    "coder": "You are a programmer. Write a short, runnable Python script that demonstrates: {task}",
}


class WorkerRouteV2(BaseModel):
    worker: Literal["researcher", "summarizer", "analyst", "coder"]
    rationale: str


SUPERVISOR_PROMPT_V2 = (
    "You are a supervisor. Classify this task to the best worker: "
    "researcher (needs facts/research), summarizer (needs condensed info), "
    "analyst (needs structured analysis), coder (needs working code). Task: {task}"
)

router_llm_v2 = llm.with_structured_output(WorkerRouteV2)


def make_worker_v2(worker_type: str):
    def worker(state: SupervisorState) -> dict:
        prompt = WORKER_PROMPTS_V2[worker_type].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"worker_result": result.content}
    worker.__name__ = f"{worker_type}_worker"
    return worker


def supervisor_v2(state: SupervisorState) -> dict:
    prompt = SUPERVISOR_PROMPT_V2.format(task=state["task"])
    result = router_llm_v2.invoke([HumanMessage(content=prompt)])
    print(f"  [supervisor] → {result.worker} | {result.rationale[:70]}")
    return {"worker_type": result.worker, "rationale": result.rationale}


def create_workflow_v2():
    graph = StateGraph(SupervisorState)
    graph.add_node("supervisor", supervisor_v2)
    for wt in WORKER_PROMPTS_V2:
        graph.add_node(wt, make_worker_v2(wt))
        graph.add_edge(wt, END)
    graph.set_entry_point("supervisor")
    graph.add_conditional_edges("supervisor", route, {wt: wt for wt in WORKER_PROMPTS_V2})
    return graph.compile()


app_v2 = create_workflow_v2()

coding_task = "Write a script that reads a CSV file and prints the first 5 rows."
result_v2 = app_v2.invoke({**EMPTY_STATE, "task": coding_task})
print(f"\nWorker: {result_v2['worker_type']}")
print(f"Result:\n{result_v2['worker_result'][:400]}")

---

### Exercise 2 Answer — Confidence Scoring

**What changes:** Add `confidence: float` to `WorkerRoute` and `SupervisorState`. Write confidence to state in the supervisor node. Update the routing function to fall back on low confidence.

**What good output looks like:** Ambiguous tasks like "Tell me something about Python" should produce `confidence < 0.8`. Clear tasks like "Analyze TypeScript vs JavaScript" should produce `confidence > 0.8`.

**Key learning:** Confidence scoring turns the routing decision from binary into a continuous signal you can monitor. Log low-confidence decisions to identify gaps in your supervisor prompt or worker coverage.

In [ ]:
# ── Exercise 2 Answer ─────────────────────────────────────────────────────────

class WorkerRouteV3(BaseModel):
    worker: Literal["researcher", "summarizer", "analyst"]
    rationale: str
    confidence: float  # 0.0 = uncertain, 1.0 = certain


class SupervisorStateV2(TypedDict):
    task: str
    worker_type: str
    rationale: str
    confidence: float
    worker_result: str


CONFIDENCE_THRESHOLD = 0.6
router_llm_v3 = llm.with_structured_output(WorkerRouteV3)


# Test on clear vs ambiguous tasks — compare confidence levels
for task in [
    "Analyze the pros and cons of GraphQL vs REST.",  # clear → analyst, high confidence
    "Tell me about computers.",                        # ambiguous, lower confidence
]:
    result = router_llm_v3.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=task))])
    flag = " *** LOW CONFIDENCE ***" if result.confidence < CONFIDENCE_THRESHOLD else ""
    print(f"Task:       {task[:60]}")
    print(f"Worker:     {result.worker}  confidence={result.confidence:.2f}{flag}")
    print(f"Rationale:  {result.rationale[:80]}")
    print()

---

## What's Next?

You now have a solid foundation in the Supervisor-Worker pattern. Here is where to go deeper:

### Related examples in this repo
- **[6-multi-agent-graph](../6-multi-agent-graph/)** — message-passing multi-agent architecture with `add_messages` reducer; agents communicate via a shared message list instead of a typed state dict
- **[19-multi-agent-notebook](../19-multi-agent-notebook/)** — multi-agent coordination patterns with detailed notebook explanations
- **[38-langgraph-command-pattern](../38-langgraph-command-pattern/)** — `Command(goto=route)` as an alternative routing primitive; compare its trade-offs against `with_structured_output`

### Go deeper on routing
- **Tool-calling routing** — instead of `with_structured_output`, bind tools to the supervisor so it can call `route_to_researcher()` as a function call
- **Semantic routing** — embed task descriptions and worker descriptions, route by cosine similarity (no LLM call needed for routing)
- **LangGraph Studio** — visual debugger for multi-agent graphs: https://studio.langchain.com

### Production considerations
- **Observability** — enable LangSmith tracing to log every supervisor decision, rationale, and worker output
- **Async** — all LangGraph graphs support `ainvoke()` for non-blocking concurrent execution
- **Checkpointing** — add a `MemorySaver` checkpointer to persist state across interrupted runs

### Academic references
> Wu, Q., Bansal, G., Zhang, J., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* arXiv:2308.08155. https://arxiv.org/abs/2308.08155
>
> Park, J. S., O'Brien, J. C., Cai, C. J., Morris, M. R., Liang, P., & Bernstein, M. S. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442
>
> LangGraph multi-agent documentation: https://langchain-ai.github.io/langgraph/concepts/multi_agent/
>
> Pydantic structured output guide: https://python.langchain.com/docs/concepts/structured_outputs/

---

*Workshop complete. The natural next step is example 38 — compare `Command(goto=route)` with the structured output approach you just learned.*